# 13 Fairness vs Accuracy Tradeoff

This notebook focuses on the central ethical question: how much model performance changes when fairness interventions are introduced.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

ROOT = Path.cwd()
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
DATA_PATH = next((ROOT / 'data' / 'raw').glob('*'))
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)

def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

tradeoff = pd.read_csv(REPORTS / 'fairness_reports' / 'application_model' / 'xgboost_application_fairness_accuracy_tradeoff.csv')
summary = load_json(REPORTS / 'fairness_reports' / 'application_model' / 'xgboost_application_bias_mitigation_summary.json')
display(tradeoff)


## Baseline vs mitigation

The baseline application model starts from moderate fairness gaps and useful predictive performance. Reweighing and post-processing show that fairness mitigation is a tradeoff problem rather than a free gain.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=tradeoff, x='method', y='perf_roc_auc', ax=axes[0], hue='method', palette='Blues_d', legend=False)
axes[0].set_title('ROC-AUC by Method')
sns.barplot(data=tradeoff, x='method', y='fair_demographic_parity_difference', ax=axes[1], hue='method', palette='Reds_d', legend=False)
axes[1].set_title('Demographic Parity Difference by Method')
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()


In [ ]:
melted = tradeoff.melt(id_vars='method', value_vars=['fair_demographic_parity_difference', 'fair_equalized_odds_difference', 'fair_equal_opportunity_difference'], var_name='metric', value_name='value')
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=melted, x='metric', y='value', hue='method', ax=ax)
ax.set_title('Fairness Metrics Before and After Mitigation')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()


## Ethical interpretation

The saved results show that post-processing reduces demographic parity difference more than the baseline, but it also lowers ROC-AUC and recall. That is the correct way to present fairness mitigation in a CV-ready project: it is a governance decision with measurable predictive cost, not a cosmetic add-on.
